# Face 05 · 1:1 Verification Evaluation

Measures what check-in needs: **genuine vs impostor cosine scores**, ROC, and the threshold where
**FAR = 0.1 %** – the spec requires **FRR ≤ 3 %** at that point.
Also: per-group FRR (gender, age band, skin tone, glasses / cap / mask) on the consented in-house set,
and the gap between the phone model and the server teacher.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from Common_Utils.Metrics_Utils import roc_points, threshold_at_far

@torch.no_grad()
def embed(model, images):            # images: N x 3 x 112 x 112 in [-1, 1]
    e = model(images) + model(torch.flip(images, dims=[3]))   # flip test-time augmentation
    return F.normalize(e)

def pair_scores(emb_a, emb_b):
    return (emb_a * emb_b).sum(1).cpu().numpy()

# genuine = pair_scores(...same person pairs...); impostor = pair_scores(...different people...)
genuine, impostor = np.random.normal(0.6, 0.1, 6000), np.random.normal(0.05, 0.1, 6000)   # placeholder demo data
thr, far, frr = threshold_at_far(genuine, impostor, 1e-3)
print(f"threshold={thr:.3f}  FAR={far:.4%}  FRR={frr:.2%}  -> target FRR <= 3%")

In [ ]:
t, FAR, FRR = roc_points(genuine, impostor)
plt.figure(figsize=(5, 4)); plt.semilogx(FAR + 1e-6, 1 - FRR); plt.axvline(1e-3, ls="--")
plt.xlabel("FAR"); plt.ylabel("TAR"); plt.title("Face verification ROC"); plt.grid(True, which="both", alpha=.3)